In [1]:
import re
import pandas as pd
import numpy as np
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

In [15]:
## Load data from pickled files
df = pd.read_pickle("pos_sent.pkl")
df.drop_duplicates(subset='COMMENT', inplace=True)
df.head()

,GAME_ID,GAME_NAME,COMMENT,RATING,CMT_LEN,CMT_SENT,KEY_PHRASES
1,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,I was favorably surprised by this I didnt buy ...,8.5,792,0.9751,"[consider somewhat mid im, also strikingly wel..."
2,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,My favorite in the Pandemic Legacy series and ...,8.5,318,0.9096,"[ive bought another copy, favorite legacy game..."
3,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,My favorite coop game,8.5,23,0.4588,[favorite coop game]
4,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,This is a must buy legacy game for anyone that...,8.5,131,0.5106,"[must buy legacy game, cleanest legacy games, ..."
5,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,First complete play through was awesome played...,8.5,97,0.9186,"[first complete play, still great, campaign 2,..."


In [16]:
## Remove common noice in natural language
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    if len(text) > 50:
        return text
    
## Create list of documents (comments)
documents = np.array(df['COMMENT'])

cleaned_docs = np.vectorize(clean_text)(documents)
cleaned_docs = np.array([x for x in cleaned_docs if x != "None"])

In [17]:
custom_stopwords = list(ENGLISH_STOP_WORDS.union({
    "i", "favorite", "in", "game", "great", "good", "fun", "like", "really", "best", "ive", "games",
    "love", "absolutely", "just", "playing", "players", "gloomhaven", "pandemic", "played", "times",
    "far", "legacy", "coop", "amazing", "simply", "legacy", "better", "lot", "dont", "new", "experience",
    "awesome", "gaming", "fantastic", "season", "need", "want", "wait", "people", "friends", "different",
    "player", "excellent", "based", "rating", "perfect", "close", "introduction", "pretty", "set", "im",
    "spirits", "way", "group", "takes", "bit", "handed", "enjoy", "exclusively", "wife", "wish", "copy",
    "willing", "week", "worth", "table", "second", "gets", "having", "probably", "experiences", "recommend",
    "greatest", "truly", "enjoyable", "got", "bought", "usually", "able", "highly", "dd", "favourite", "rpg",
    "recommended", "date", "gamers", "blast", "getting", "think", "id", "family", "buy", "hard", "regular",
    "regularly", "works", "chance", "rated", "wanting", "difficult", "took", "know", "started", "right", "did",
    "excited", "wanted", "day", "waiting", "sure", "days", "year", "years", "joy", "higher", "review", "cards", 
    "board", "plays", "feel", "play", "true", "brain", "spirit", "island", "downside", "trying", "tear", "thing"
}))

In [18]:
## Create TfIdf vectorizer and fit_transform it to the comments
vectorizer = TfidfVectorizer(
    max_df=0.95,
    min_df=10,
    stop_words=custom_stopwords,
    ngram_range=(1,1)
)
TfIdf_matrix = vectorizer.fit_transform(cleaned_docs)

## Create the NMF Model for topic analysis
model = NMF(n_components=3, random_state=42)
W = model.fit_transform(TfIdf_matrix)
H = model.components_

In [19]:
## Get the top keywords in each topic
feature_names = vectorizer.get_feature_names_out()

for topic_idx, topic in enumerate(H):
    top_words = [feature_names[i] for i in topic.argsort()[:-11:-1]]
    print(f"Topic #{topic_idx + 1}: {' | '.join(top_words)}")

Topic #1: story | campaign | scenarios | theme | mechanics | rules | gameplay | easy | characters | interesting
Topic #2: solo | multiplayer | mode | thematic | replayability | long | theme | complex | puzzle | mage
Topic #3: time | long | setup | hours | rules | learn | spend | investment | spent | come


# Topic Keywords
## Topic 1
 - campaign | story | scenarios | rules | mechanics | characters | gameplay | theme | scenario | interesting
## Topic 2
 - solo | multiplayer | mode | thematic | replayability | long | theme | complex | puzzle | mage
## Topic 3
 - time | long | setup | hours | investment | spend | consuming | space | learn | spent

# Topic Labels
- Topic 1: Narrative-Driven Gameplay Experience
- Topic 2: Thoughtful & Thematic Solo Play
- Topic 3: Time & Commitment Satisfaction

# Strategic Questions

| **Question**                                                       | **What to Do**                                                        |
| ------------------------------------------------------------------ | --------------------------------------------------------------------- |
| Which design elements do players find most immersive and engaging? | Focus on reviews weighted toward the *narrative-driven* topic.        |
| What makes long or complex games still feel satisfying to players? | Extract reviews from the *time-intensive but rewarding* topic.        |
| How do players describe their favorite solo game experiences?      | Sample *solo play* topic reviews for puzzle-like, challenging praise. |


In [ ]:
## Match reviews to topics
X_tfidf = vectorizer.transform(cleaned_docs)
W = model.transform(X_tfidf)

## Get best topic and match strength
best_topic = W.argmax(axis=1)
best_strength = W.max(axis=1)

## Create data frame with matched topics
matched_reviews = pd.DataFrame({"review": cleaned_docs, "topic": best_topic, "match_strength": best_strength})
topic_labels = {
    0: "Narrative-Drive Gameplay Experience",
    1: "Thoughtful & Thematic Solo Play",
    2: "Time & Commitment Satisfaction"
}

matched_reviews['topic'] = matched_reviews['topic'].map(topic_labels)
matched_reviews = matched_reviews.sort_values('match_strength', ascending=False)

## Show the data frame
matched_reviews

# Topic 1: Example Reviews

In [23]:
topic_1 = matched_reviews[matched_reviews['topic'] == "Narrative-Drive Gameplay Experience"]
topic_1.head()

,review,topic,match_strength
12266,this entry reflects the base game logs in this...,Narrative-Drive Gameplay Experience,0.090957
18018,though this is not the prototypical game that ...,Narrative-Drive Gameplay Experience,0.085261
15430,great implementation of theme the players char...,Narrative-Drive Gameplay Experience,0.080499
1663,gloomhaven is by far one of the best and immer...,Narrative-Drive Gameplay Experience,0.080143
12324,i want to enjoy this game more however the cos...,Narrative-Drive Gameplay Experience,0.078525


# Topic 2: Example Reviews

In [ ]:
topic_2 = matched_reviews[matched_reviews['topic'] == "Thoughtful & Thematic Solo Play"]
topic_2.head()

,review,topic,match_strength
10397,good game ive played solo but would probably e...,Thoughtful & Thematic Solo Play,0.282438
9379,one of the best games ive ever played perfect ...,Thoughtful & Thematic Solo Play,0.282438
9752,the more i play this the more i like it great ...,Thoughtful & Thematic Solo Play,0.282438
2306,wanted to love it not a great solo experience ...,Thoughtful & Thematic Solo Play,0.282438
6958,really great even solo cant wait to play it wi...,Thoughtful & Thematic Solo Play,0.282438


# Topic 3: Example Reviews

In [25]:
topic_3 = matched_reviews[matched_reviews['topic'] == "Time & Commitment Satisfaction"]
topic_3.head()

,review,topic,match_strength
15365,probably my most played game one of my favorit...,Time & Commitment Satisfaction,0.272986
15426,like it a lot so far need more time with it be...,Time & Commitment Satisfaction,0.272986
6206,this game is so fking good you will want to pl...,Time & Commitment Satisfaction,0.272986
6183,only played once so far was fun but best i can...,Time & Commitment Satisfaction,0.272986
21422,its good we just find ourselves playing other ...,Time & Commitment Satisfaction,0.272986
